Day 4 file of week 1


In [ ]:
import pandas as pd

# Task 1, reading a csv, reading it's shape and it's columns and dtypes
df = pd.read_csv("market_data.csv")
print(df.shape)
columns = [column for column in df.columns]
print(columns)
print(df.dtypes)

(8523, 12)
['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Outlet_Sales']
Item_Identifier                  str
Item_Weight                  float64
Item_Fat_Content                 str
Item_Visibility              float64
Item_Type                        str
Item_MRP                     float64
Outlet_Identifier                str
Outlet_Establishment_Year      int64
Outlet_Size                      str
Outlet_Location_Type             str
Outlet_Type                      str
Item_Outlet_Sales            float64
dtype: object


In [ ]:
# Task 2, count missing values per column and handle them
# missing values per column
from pydantic import BaseModel

print(df.isnull().sum())
# First drop all duplicates
df.drop_duplicates(inplace=True)
# empty values are only part of the item weight and outlet size
# the item weight is to be set to the mean item weight so the average doesn't change
# the outlet size is to be set to the most common

# first get the mean item weight

mean_value = float(format(df["Item_Weight"].mean(), ".2f"))
# Assign the na values
df["Item_Weight"] = df["Item_Weight"].fillna(mean_value)

# Now we will get the most common value

outlet_sizes = df.groupby("Outlet_Size").size()


class MostCommonValue(BaseModel):
    amount: int
    type: str


most_common_value: MostCommonValue = MostCommonValue(amount=0, type="")
for outlet_size, size in outlet_sizes.items():
    if most_common_value.amount < size:
        most_common_value.amount = size
        most_common_value.type = str(outlet_size)
# Assign the most common type to be the one for the rest of NA one's
df["Outlet_Size"] = df["Outlet_Size"].fillna(most_common_value.type)

# Filter the data to a meaninful subset and extract meaningful stat

items_in_supermarket_1 = df[df["Outlet_Type"] == "Supermarket Type1"]
items_in_supermarket_2 = df[df["Outlet_Type"]=="Supermarket Type2"]
most_common_item_type_in_supermarket_1 = MostCommonValue(amount=0, type="")
for item_type, size in (
    items_in_supermarket_1.groupby("Item_Type").size().items()
):
    if most_common_item_type_in_supermarket_1.amount < size:
        most_common_item_type_in_supermarket_1 = MostCommonValue(
            amount=size, type=str(item_type)
        )
most_common_item_type_in_supermarket_2 = MostCommonValue(amount=0, type="")
for outlet_type, size in items_in_supermarket_2.groupby("Item_Type").size().items():
    if most_common_item_type_in_supermarket_2.amount < size:
        most_common_item_type_in_supermarket_2 = MostCommonValue(
            amount=size, type=str(outlet_type)
        )
# Here it shows that the most common items in both supermarket types is fruits and vegetables but supermarket 1 has way more of them, 8x as much
print(
    most_common_item_type_in_supermarket_1.type,
    most_common_item_type_in_supermarket_1.amount,
    most_common_item_type_in_supermarket_2.type,
    most_common_item_type_in_supermarket_2.amount,
)



Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64
Fruits and Vegetables 805 Fruits and Vegetables 135
